# Phase 1: Preprocessing Pipeline Verification

This notebook runs the preprocessing pipeline for both the German Credit and GMSC datasets, verifies that the preprocessing parameters were fit correctly without data leakage, inspects the preprocessed outputs, and outputs a summary of the data properties.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib

# Add root folder to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))

from src.preprocessing.pipeline import preprocess_german_credit, preprocess_gmsc, download_german_credit, sample_gmsc

## 1. Run Preprocessing Pipeline

We start by executing the preprocessing functions. For the German Credit dataset, it will automatically download the OpenML source if it is not present in `data/raw/`. For the GMSC dataset, it requires the raw file `data/raw/cs-training.csv` to be present.

In [2]:
# Run German Credit preprocessing
preprocess_german_credit(raw_path="../../data/raw/german_credit.csv", processed_dir="../../data/processed/", model_dir="../../models/baseline/")

Preprocessing German Credit dataset...
German Credit dataset preprocessing completed successfully!


In [3]:
# Check if Kaggle file exists, and run GMSC preprocessing if so
gmsc_raw_path = "../../data/raw/cs-training.csv"
if os.path.exists(gmsc_raw_path):
    preprocess_gmsc(raw_path="../../data/raw/gmsc_sampled_10k.csv", processed_dir="../../data/processed/", model_dir="../../models/baseline/")
else:
    print(f"GMSC raw file '{gmsc_raw_path}' is missing. Skipping preprocessing execution for GMSC in this cell.")
    print("Please place the downloaded Kaggle file in data/raw/ to run this fully.")

Preprocessing GMSC dataset...


GMSC dataset preprocessing completed successfully!


## 2. Verify Preprocessing and Assertions (Leakage Prevention)

We load the saved splits and fitted objects to perform validation assertions:
* Confirm class balance is preserved between train and test sets.
* Assert that the scaling parameter values were computed *only* on the training splits (i.e. no leakage).

In [4]:
# Verification for German Credit
X_train_g = pd.read_csv("../../data/processed/X_train_german.csv")
y_train_g = pd.read_csv("../../data/processed/y_train_german.csv")
X_test_g = pd.read_csv("../../data/processed/X_test_german.csv")
y_test_g = pd.read_csv("../../data/processed/y_test_german.csv")

print(f"German Credit Train shape: {X_train_g.shape}, Test shape: {X_test_g.shape}")

# Class balance check
train_pos_ratio = y_train_g.mean().values[0]
test_pos_ratio = y_test_g.mean().values[0]
print(f"Train positive class ratio (encoded target): {train_pos_ratio:.4f}")
print(f"Test positive class ratio (encoded target): {test_pos_ratio:.4f}")
# Difference should be minimal (typically < 0.01)
assert abs(train_pos_ratio - test_pos_ratio) < 0.02, "German Credit stratified split class ratio mismatch!"

# Leakage check: scaled numeric features in X_train_g should have mean ~0 and std ~1
scaler_g = joblib.load("../../models/baseline/german_scaler.joblib")
num_cols_g = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']

for col in num_cols_g:
    mean_val = X_train_g[col].mean()
    std_val = X_train_g[col].std(ddof=0)
    print(f"German Credit Train scaled {col} - Mean: {mean_val:.4f}, Std: {std_val:.4f}")
    assert abs(mean_val) < 1e-7, f"Leakage detected: Mean of {col} in train is not 0!"
    assert abs(std_val - 1.0) < 1e-7, f"Leakage detected: Std of {col} in train is not 1!"

print("\nGerman Credit Preprocessing Validation Passed!")

German Credit Train shape: (700, 20), Test shape: (300, 20)
Train positive class ratio (encoded target): 0.3000
Test positive class ratio (encoded target): 0.3000
German Credit Train scaled duration - Mean: 0.0000, Std: 1.0000
German Credit Train scaled credit_amount - Mean: -0.0000, Std: 1.0000
German Credit Train scaled installment_commitment - Mean: -0.0000, Std: 1.0000
German Credit Train scaled residence_since - Mean: 0.0000, Std: 1.0000
German Credit Train scaled age - Mean: -0.0000, Std: 1.0000
German Credit Train scaled existing_credits - Mean: -0.0000, Std: 1.0000
German Credit Train scaled num_dependents - Mean: 0.0000, Std: 1.0000

German Credit Preprocessing Validation Passed!


In [5]:
# Verification for GMSC if splits are available
if os.path.exists("../../data/processed/X_train_gmsc.csv"):
    X_train_m = pd.read_csv("../../data/processed/X_train_gmsc.csv")
    y_train_m = pd.read_csv("../../data/processed/y_train_gmsc.csv")
    X_test_m = pd.read_csv("../../data/processed/X_test_gmsc.csv")
    y_test_m = pd.read_csv("../../data/processed/y_test_gmsc.csv")

    print(f"GMSC Train shape: {X_train_m.shape}, Test shape: {X_test_m.shape}")

    # Class balance check
    train_pos_ratio_m = y_train_m.mean().values[0]
    test_pos_ratio_m = y_test_m.mean().values[0]
    print(f"Train positive class ratio: {train_pos_ratio_m:.4f}")
    print(f"Test positive class ratio: {test_pos_ratio_m:.4f}")
    assert abs(train_pos_ratio_m - test_pos_ratio_m) < 0.02, "GMSC stratified split class ratio mismatch!"

    # Leakage check
    scaler_m = joblib.load("../../models/baseline/gmsc_scaler.joblib")
    for col in X_train_m.columns:
        mean_val = X_train_m[col].mean()
        std_val = X_train_m[col].std(ddof=0)
        print(f"GMSC Train scaled {col} - Mean: {mean_val:.4f}, Std: {std_val:.4f}")
        assert abs(mean_val) < 1e-7, f"Leakage detected: Mean of {col} in GMSC train is not 0!"
        assert abs(std_val - 1.0) < 1e-7, f"Leakage detected: Std of {col} in GMSC train is not 1!"

    print("\nGMSC Preprocessing Validation Passed!")
else:
    print("GMSC processed splits do not exist. Skipping GMSC validation tests.")

GMSC Train shape: (7000, 10), Test shape: (3000, 10)
Train positive class ratio: 0.0669
Test positive class ratio: 0.0667
GMSC Train scaled RevolvingUtilizationOfUnsecuredLines - Mean: 0.0000, Std: 1.0000
GMSC Train scaled age - Mean: -0.0000, Std: 1.0000
GMSC Train scaled NumberOfTime30-59DaysPastDueNotWorse - Mean: -0.0000, Std: 1.0000
GMSC Train scaled DebtRatio - Mean: -0.0000, Std: 1.0000
GMSC Train scaled MonthlyIncome - Mean: 0.0000, Std: 1.0000
GMSC Train scaled NumberOfOpenCreditLinesAndLoans - Mean: -0.0000, Std: 1.0000
GMSC Train scaled NumberOfTimes90DaysLate - Mean: 0.0000, Std: 1.0000
GMSC Train scaled NumberRealEstateLoansOrLines - Mean: -0.0000, Std: 1.0000
GMSC Train scaled NumberOfTime60-89DaysPastDueNotWorse - Mean: 0.0000, Std: 1.0000
GMSC Train scaled NumberOfDependents - Mean: 0.0000, Std: 1.0000

GMSC Preprocessing Validation Passed!


## 3. Preprocessing Summary Table

We compile a summary of the characteristics of both datasets before and after preprocessing.

In [6]:
summary_data = []

# German Credit Stats
raw_german = pd.read_csv("../../data/raw/german_credit.csv")
summary_data.append({
    "Dataset": "German Credit",
    "Original Rows": len(raw_german),
    "Train Rows (70%)": len(X_train_g),
    "Test Rows (30%)": len(X_test_g),
    "Features": X_train_g.shape[1],
    "Target Balance (Train)": f"{train_pos_ratio*100:.2f}% / {(1-train_pos_ratio)*100:.2f}%",
    "Target Balance (Test)": f"{test_pos_ratio*100:.2f}% / {(1-test_pos_ratio)*100:.2f}%"
})

# GMSC Stats if present
if os.path.exists("../../data/processed/X_train_gmsc.csv"):
    raw_gmsc = pd.read_csv("../../data/raw/gmsc_sampled_10k.csv")
    summary_data.append({
        "Dataset": "Give Me Some Credit",
        "Original Rows": len(raw_gmsc),
        "Train Rows (70%)": len(X_train_m),
        "Test Rows (30%)": len(X_test_m),
        "Features": X_train_m.shape[1],
        "Target Balance (Train)": f"{train_pos_ratio_m*100:.2f}% / {(1-train_pos_ratio_m)*100:.2f}%",
        "Target Balance (Test)": f"{test_pos_ratio_m*100:.2f}% / {(1-test_pos_ratio_m)*100:.2f}%"
    })
else:
    summary_data.append({
        "Dataset": "Give Me Some Credit",
        "Original Rows": "Missing cs-training.csv",
        "Train Rows (70%)": "N/A",
        "Test Rows (30%)": "N/A",
        "Features": "N/A",
        "Target Balance (Train)": "N/A",
        "Target Balance (Test)": "N/A"
    })

df_summary = pd.DataFrame(summary_data)
from IPython.display import display, HTML
display(HTML(df_summary.to_html(index=False)))

Dataset,Original Rows,Train Rows (70%),Test Rows (30%),Features,Target Balance (Train),Target Balance (Test)
German Credit,1000,700,300,20,30.00% / 70.00%,30.00% / 70.00%
Give Me Some Credit,10000,7000,3000,10,6.69% / 93.31%,6.67% / 93.33%
